In [90]:
#!/usr/bin/env python
# coding: utf-8

# # Simulating different telescopes
# This notebooks provides examples in how to use the lenstronomy.SimulationAPI modules in simulating (realistic) mock lenses taylored to a specific observation and instrument and makes a montage of different telescope settings currently available.
# 
# The module enables to use the astronomical magnitude conventions and can translate those into the lenstronomy core module configurations.

# In[1]:


import copy
import h5py
import os
import numpy as np
import scipy
import sncosmo
from PIL import Image
import galsim
from astropy.cosmology import WMAP9 as cosmo
import matplotlib.pyplot as plt

from astropy.table import Table
# make sure lenstronomy is installed, otherwise install the latest pip version
try:
    import lenstronomy
except:
    get_ipython().system('pip install lenstronomy')
from astropy.io import fits
from astropy import wcs
import galsim

# lenstronomy module import
from lenstronomy.Util import image_util, data_util, util
import lenstronomy.Plots.plot_util as plot_util
from lenstronomy.SimulationAPI.sim_api import SimAPI
from lenstronomy.Plots.plot_util import coordinate_arrows, scale_bar
from lenstronomy.SimulationAPI.point_source_variability import PointSourceVariability

from astropy.table import Table
import sncosmo

### Get supernova lightcurve magnitudes

In [91]:
def var_func(time, zero_point, band, z_source): #band either desg, desi, desr
    if z_source > 0.915:
        z_source = 0.915
    
    obs = Table({'time': [time],
             'band': [band],          #filters we are observing in
             'gain': [1.0],              #dw
             'skynoise': [0],           #depends on exposure time, but will be addig noise later in lenstronomy, could set to 0
             'zp': [zero_point],        # zero point, what corresponds to 0 flux in this image - pixel values to magnitudes
             'zpsys':['ab']})           # ab magnitudes, units of zero point

    model = sncosmo.Model(source='salt2')
    params = {'z': 0, 't0': 56200.0, 'x0':1.e-5, 'x1': 0.1, 'c': -0.1}

    lcs = sncosmo.realize_lcs(obs, model, [params])
    flux = np.array(lcs[0]['flux'])
    # print(lcs[0]['time'])
    # print(flux)
    zero_points = np.array(obs['zp'])
    magnitude = zero_points - 2.5 * np.log10(flux)
    # print(time,magnitude)
    # return flux
    return magnitude

var_func_g = lambda t: var_func(t, 28.30, 'lsstg', z_source)
var_func_r = lambda t: var_func(t, 28.13, 'lsstr', z_source)
var_func_i = lambda t: var_func(t, 27.7, 'lssti', z_source)

### Def plot image

In [92]:
def simulate_rgb(ConfigList, size, kwargs_numerics, time, source_x=0,source_y=0,returnsqrt = False):
    band_g, band_r, band_i = ConfigList
    kwargs_i_band = band_i.kwargs_single_band()
    kwargs_g_band = band_g.kwargs_single_band()
    kwargs_r_band = band_r.kwargs_single_band()
    
    # print(kwargs_g_band) #'pixel_scale' : kwargs_b_band['pixel_scale']
    # set number of pixels from pixel scale
    pixel_scale = kwargs_g_band['pixel_scale']    
    numpix = int(round(size / pixel_scale))
    # kwargs_model['pixel_scale'] = kwargs_g_band['pixel_scale']
    
    sim_i = SimAPI(numpix=numpix, kwargs_single_band=kwargs_i_band, kwargs_model=kwargs_model_time_var)
    sim_g = SimAPI(numpix=numpix, kwargs_single_band=kwargs_g_band, kwargs_model=kwargs_model_time_var)
    sim_r = SimAPI(numpix=numpix, kwargs_single_band=kwargs_r_band, kwargs_model=kwargs_model_time_var)

    ###### Fix error when trying to get PSF sizes
    ## This deosnt work ugh
    # print("pixel scale:", pixel_scale)
    # sim_g.psf_class.set_pixel_size(pixel_scale)

    # print("psf pixel_size:", sim_g.psf_class._pixel_size)
    # print("psf fwhm:", sim_g.psf_class._fwhm)
    # print("psf truncation:", sim_g.psf_class._truncation)

    

    # return the ImSim instance. With this class instance, you can compute all the
    # modelling accessible of the core modules. See class documentation and other notebooks.
    imSim_i = sim_i.image_model_class(kwargs_numerics)
    imSim_g = sim_g.image_model_class(kwargs_numerics)
    imSim_r = sim_r.image_model_class(kwargs_numerics)


    # turn magnitude kwargs into lenstronomy kwargs
    kwargs_lens_light_g, kwargs_source_g, kwargs_ps_g = sim_g.magnitude2amplitude(kwargs_lens_light_mag_g, kwargs_source_mag_g, kwargs_ps_mag_g)
    kwargs_lens_light_r, kwargs_source_r, kwargs_ps_r = sim_g.magnitude2amplitude(kwargs_lens_light_mag_r, kwargs_source_mag_r, kwargs_ps_mag_r)
    kwargs_lens_light_i, kwargs_source_i, kwargs_ps_i = sim_r.magnitude2amplitude(kwargs_lens_light_mag_i, kwargs_source_mag_i, kwargs_ps_mag_i)

    ps_var_g = PointSourceVariability(source_x, source_y, var_func_g, numpix, kwargs_g_band, kwargs_model_time_var, kwargs_numerics,
                 kwargs_lens, kwargs_source_mag_g, kwargs_lens_light_mag_g, kwargs_ps_mag=kwargs_ps_mag_g)

    ps_var_r = PointSourceVariability(source_x, source_y, var_func_r, numpix, kwargs_r_band, kwargs_model_time_var, kwargs_numerics,
                 kwargs_lens, kwargs_source_mag_r, kwargs_lens_light_mag_r, kwargs_ps_mag=kwargs_ps_mag_r)

    ps_var_i = PointSourceVariability(source_x, source_y, var_func_i, numpix, kwargs_i_band, kwargs_model_time_var, kwargs_numerics,
                 kwargs_lens, kwargs_source_mag_i, kwargs_lens_light_mag_i, kwargs_ps_mag=kwargs_ps_mag_i)
    
    delays = ps_var_g.delays
    print('DELAYS',delays,np.max(delays))

    image_g = ps_var_g.image_time(time=time)#imSim_b.image(kwargs_lens, kwargs_source_g, kwargs_lens_light_g, kwargs_ps_g)
    image_r = ps_var_r.image_time(time=time)#imSim_g.image(kwargs_lens, kwargs_source_r, kwargs_lens_light_r, kwargs_ps_r)
    image_i = ps_var_i.image_time(time=time)#imSim_r.image(kwargs_lens, kwargs_source_i, kwargs_lens_light_i, kwargs_ps_i)

    # add noise
    image_i += sim_i.noise_for_model(model=image_i)
    image_g += sim_g.noise_for_model(model=image_g)
    image_r += sim_r.noise_for_model(model=image_r)

    # and plot it

    img = np.zeros((image_g.shape[0], image_g.shape[1], 3), dtype=float)
    #scale_max=10000
    def _scale_max(image): 
        flat=image.flatten()
        flat.sort()
        scale_max = flat[int(len(flat)*0.95)]
        return scale_max
    if returnsqrt==True:
        img[:,:,0] = plot_util.sqrt(image_g, scale_min=0, scale_max=_scale_max(image_g))
        img[:,:,1] = plot_util.sqrt(image_r, scale_min=0, scale_max=_scale_max(image_r))
        img[:,:,2] = plot_util.sqrt(image_i, scale_min=0, scale_max=_scale_max(image_i))
    else:
        img[:,:,0] = image_i
        img[:,:,1] = image_g
        img[:,:,2] = image_r
        
    data_class = sim_i.data_class
    # #print(sim_r.psf_class.fwhm)
    # psf_i = sim_i.psf_class.kernel_point_source
    # psf_g = sim_g.psf_class.kernel_point_source
    # psf_r = sim_r.psf_class.kernel_point_source
    # psf = np.stack((psf_i,psf_g,psf_r))

    # EXTRACT PSF KERNELS (this is what you need!)
    psf_i = sim_i.psf_class.kernel_point_source
    psf_g = sim_g.psf_class.kernel_point_source
    psf_r = sim_r.psf_class.kernel_point_source
    
    # Get PSF size
    psf_shape_i = psf_i.shape
    psf_shape_g = psf_g.shape
    psf_shape_r = psf_r.shape
    
    print(f"PSF shapes - g: {psf_shape_g}, r: {psf_shape_r}, i: {psf_shape_i}")
    
    
    
    return img, data_class, [ps_var_g, ps_var_r, ps_var_i]



### Camera kwargs

In [93]:
nbands = 3
nepochs = 15
bands = ['lsstg', 'lsstr', 'lssti']

times = np.arange(56100,56350,2)

# ## Define camera and observations
# As an example, we define the camera and observational settings of a LSST-like observation. We define one camera setting and three different observations corresponding to g,r,i imaging.
# 
# For the complete list of possible settings, we refer to the SimulationAPI.observation_api classes. There are pre-configured settings which approximately mimic observations from current and future instruments. Be careful using those and check whether they are sufficiently accurate for your specific science case!

# In[3]:


# Instrument setting from pre-defined configurations

from lenstronomy.SimulationAPI.ObservationConfig.DES import DES
#from lenstronomy.SimulationAPI.ObservationConfig.LSSTsingleobs import LSST
from lenstronomy.SimulationAPI.ObservationConfig.LSST import LSST

from lenstronomy.SimulationAPI.ObservationConfig.Euclid import Euclid
from lenstronomy.SimulationAPI.ObservationConfig.Roman import Roman

DES_g = DES(band='g', psf_type='GAUSSIAN', coadd_years=3)
DES_r = DES(band='r', psf_type='GAUSSIAN', coadd_years=3)
DES_i = DES(band='i', psf_type='GAUSSIAN', coadd_years=3)
des = [DES_g, DES_r, DES_i]

LSST_g = LSST(band='g', psf_type='GAUSSIAN', coadd_years=10)
LSST_r = LSST(band='r', psf_type='GAUSSIAN', coadd_years=10)
LSST_i = LSST(band='i', psf_type='GAUSSIAN', coadd_years=10)
lsst = [LSST_g, LSST_r, LSST_i]


Roman_g = Roman(band='F062', psf_type='GAUSSIAN', survey_mode='wide_area')
Roman_r = Roman(band='F106', psf_type='GAUSSIAN', survey_mode='wide_area')
Roman_i = Roman(band='F184', psf_type='GAUSSIAN', survey_mode='wide_area')#'single_exposure')#'wide_area')
roman = [Roman_g, Roman_r, Roman_i]

# lenstronomy provides these setting to be imported with the SimulationAPI.ObservationConfig routines.

kwargs_g_band_LS4 = util.merge_dicts(LS4_camera, LS4_g_band_obs)
kwargs_r_band_LS4 = util.merge_dicts(LS4_camera, LS4_r_band_obs)
kwargs_i_band_LS4 = util.merge_dicts(LS4_camera, LS4_i_band_obs)
kwargs_LS4 = [kwargs_g_band_LS4,kwargs_r_band_LS4,kwargs_i_band_LS4]


import imageio.v2 as imageio
from scipy.ndimage import gaussian_filter
from lenstronomy.Cosmo.micro_lensing import einstein_radius
# find path to data
path = os.getcwd()
dirpath, _ = os.path.split(path)
module_path, _ = os.path.split(dirpath)

In [94]:
def print_telescope_params(telescope_config, name="Telescope"):
    # This extracts the full dictionary of settings lenstronomy uses internally
    kwargs = telescope_config.kwargs_single_band()
    
    print(f"--- {name} Configuration ---")
    print(f"{'Parameter':<25} | {'Value'}")
    print("-" * 40)
    for key, value in kwargs.items():
        print(f"{key:<25} | {value}")
    print("-" * 40)
    print("\n")

# --- EXAMPLES OF HOW TO CALL IT ---

# 1. For Rubin/LSST (using the object you already created)
# Assuming you have: LSST_g = LSST(band='g', ...)
print_telescope_params(LSST_r, "LSST r-band")

# 2. For Roman (using the object you already created)
# Assuming you have: Roman_g = Roman(band='F062', ...)
print_telescope_params(Roman_r, "Roman F106")


# 3. For your custom LS4
# Since LS4 isn't a lenstronomy Class but a dictionary you made manually,
# you can just print the dictionary directly:
print("--- LS4 Custom Config ---")
for key, value in kwargs_g_band_LS4.items():
    print(f"{key:<25} | {value}")

--- LSST r-band Configuration ---
Parameter                 | Value
----------------------------------------
read_noise                | 10
pixel_scale               | 0.2
ccd_gain                  | 2.3
exposure_time             | 15.0
sky_brightness            | 21.2
magnitude_zero_point      | 28.13
num_exposures             | 460
seeing                    | 0.73
psf_type                  | GAUSSIAN
----------------------------------------


--- Roman F106 Configuration ---
Parameter                 | Value
----------------------------------------
read_noise                | 15.5
pixel_scale               | 0.11
ccd_gain                  | 1
sky_brightness            | 22.99
magnitude_zero_point      | 26.44
seeing                    | 0.087
exposure_time             | 146
num_exposures             | 96
psf_type                  | GAUSSIAN
kernel_point_source       | [[3.63835275e-08 8.68507206e-08 1.38956763e-07 ... 1.17150099e-07
  6.72888589e-08 2.92235160e-08]
 [1.88676583e-08 4

In [86]:

f = h5py.File('lsst-altsched-1a-lowz.h5','r')
simitems = np.asarray(f['system']['block0_items'][()],dtype=str)
simdataall = f['system']['block0_values'][()]

# indcatalog = [96679]
indcatalog = [17462]

times = [56100]

for ind in indcatalog:
    simdata=simdataall[ind]
    z_lens = simdata[simitems == "zl"][0]
    z_source = simdata[simitems == "zs"][0]
    print('z lens',z_lens,'z source',z_source)
    kwargs_model_time_var = {'lens_model_list': ['SIE', 'SHEAR'],  # list of lens models to be used
                'lens_light_model_list': ['SERSIC_ELLIPSE'],  # list of unlensed light models to be used
                'source_light_model_list': ['SERSIC_ELLIPSE'],#['INTERPOL'],  # list of extended source models to be used, here we used the interpolated real galaxy
                'point_source_model_list': ['SOURCE_POSITION'],  # list of point source models to be used
                'z_lens': z_lens, 
                         'z_source': z_source
    } 
    Dl = cosmo.angular_diameter_distance(simdata[simitems == "zl"][0]).value
    Ds = cosmo.angular_diameter_distance(simdata[simitems == "zs"][0]).value
    Dls = cosmo.angular_diameter_distance_z1z2(simdata[simitems == "zl"][0],simdata[simitems == "zs"][0]).value
 
    theta_E_sim = simdata[simitems == "theta_e"][0]
    lens_reff = simdata[simitems == "lensgal_reff"][0]
    lens_n = simdata[simitems == "lensgal_n"][0]
    lens_theta = simdata[simitems == "lensgal_theta"][0]
    lens_ellip = simdata[simitems == "lensgal_ellip"][0]
    s = galsim.Shear(e=lens_ellip, beta=lens_theta*galsim.degrees)
    lens_e1 = s.e1
    lens_e2 = s.e2
   
    lens_gamma = simdata[simitems == "gamma"][0]
    gamma_theta = simdata[simitems == "theta_gamma"][0]
    lens_g1 = np.sin(gamma_theta*np.pi/180)*lens_gamma
    lens_g2 = np.cos(gamma_theta*np.pi/180)*lens_gamma
    lens_x = simdata[simitems == "lensgal_x"][0]
    lens_y = simdata[simitems == "lensgal_y"][0]

    host_reff = simdata[simitems == "host_reff"][0]
    host_n = simdata[simitems == "host_n"][0]
    host_theta = simdata[simitems == "host_theta"][0]
    host_ellip = simdata[simitems == "host_ellip"][0]
    try:
        sh = galsim.Shear(e=host_ellip, beta=host_theta*galsim.degrees)
    except:
        continue
    host_e1 = sh.e1
    host_e2 = sh.e2
    host_x = simdata[simitems == "host_x"][0]
    host_y = simdata[simitems == "host_y"][0]
    sn_x = simdata[simitems == "snx"][0]
    sn_y = simdata[simitems == "sny"][0]

    
    magdiff = 2.5*(np.log10(simdata[simitems == "lensgal_amplitude"][0])-np.log10(simdata[simitems == "host_amplitude"][0]))
   
    magdiffps = 2.5*(np.log10(simdata[simitems == "lensgal_amplitude"][0])-np.log10(simdata[simitems == "transient_amplitude"][0]))
  
    sig = simdata[simitems == "sigma"][0]
    theta_E = 4*np.pi*(sig/3e5)**2*Dls/Ds*180/np.pi*60*60
    M0=1.9*(sig/200)**5.1*1e8
    Mstar= 10**((np.log10(M0)-7.45)/1.05)*1e11
 
    radius = einstein_radius(Mstar, Dl*1e6, Ds*1e6)

    kwargs_lens = [
        {'theta_E': theta_E, 'e1': lens_e1, 'e2': lens_e2, 'center_x': lens_x, 'center_y': lens_y},  # SIE model
        {'gamma1': lens_g1, 'gamma2': lens_g2, 'ra_0': 0, 'dec_0': 0}  # SHEAR model
    ]
    ### lens light
    kwargs_lens_light_mag_g = [{'magnitude': 23, 'R_sersic': lens_reff, 'n_sersic': lens_n, 'e1': lens_e1, 'e2': lens_e2, 'center_x': lens_x, 'center_y': lens_y}]
    ### source light
    kwargs_source_mag_g = [{'magnitude': 23, 'R_sersic': host_reff, 'n_sersic': host_n, 'e1': host_e1, 'e2': host_e2, 'center_x': host_x, 'center_y': host_y}]

    ### point source
    kwargs_ps_mag_g = [{'magnitude': 70, 'ra_source': sn_x, 'dec_source': sn_y}]

   
    ## and now we define the colors of the other two bands

    ### r-band
    g_r_source = 1  # color mag_g - mag_r for source
    g_r_lens = -1  # color mag_g - mag_r for lens light
    g_r_ps = 0
    kwargs_lens_light_mag_r = copy.deepcopy(kwargs_lens_light_mag_g)
    kwargs_lens_light_mag_r[0]['magnitude'] -= g_r_lens

    kwargs_source_mag_r = copy.deepcopy(kwargs_source_mag_g)
    kwargs_source_mag_r[0]['magnitude'] -= g_r_source

    kwargs_ps_mag_r = copy.deepcopy(kwargs_ps_mag_g)
    kwargs_ps_mag_r[0]['magnitude'] -= g_r_ps


    # i-band
    g_i_source = 2
    g_i_lens = -2
    g_i_ps = 0
    kwargs_lens_light_mag_i = copy.deepcopy(kwargs_lens_light_mag_g)
    kwargs_lens_light_mag_i[0]['magnitude'] -= g_i_lens

    kwargs_source_mag_i = copy.deepcopy(kwargs_source_mag_g)
    kwargs_source_mag_i[0]['magnitude'] -= g_i_source

    kwargs_ps_mag_i = copy.deepcopy(kwargs_ps_mag_g)
    kwargs_ps_mag_i[0]['magnitude'] -= g_i_ps


    ## here we define the numerical options used in the ImSim module. 
    ## Have a look at the ImageNumerics class for detailed descriptions.
    ## If not further specified, the default settings are used.
    kwargs_numerics = {'point_source_supersampling_factor': 1}

    ## Hack hard coded solution to get out PSF on line 52 of ~/.conda/envs/scarlet/lib/python3.10/site-packages/lenstronomy/Data/psf.py 
    ##to get out PSF image!
    size = 6. # width of the image in units of arc seconds

    ##img_des, coords_des, sim_b = simulate_rgb(des, size=size, kwargs_numerics=kwargs_numerics)
    
    for indt,time in enumerate(times):
        print('TIME ', time)
        img_lsst, coords_lss, [ps_var_g, ps_var_r, ps_var_i] = simulate_rgb(lsst, size=size, kwargs_numerics=kwargs_numerics,time=time,source_x=sn_x,source_y=sn_y,returnsqrt=False)
        delays = ps_var_g.delays
        # print('DELAYS', delays)
        img_roman, coords_roman, [ps_var_g, ps_var_r, ps_var_i] = simulate_rgb(roman, size=size, kwargs_numerics=kwargs_numerics,time=time, source_x=sn_x,source_y=sn_y,returnsqrt=False)
        
        ##print(psf_lsst,psf_roman)
        if indt==0:
            vminlsst=np.min(img_lsst)
            vmaxlsst= 1.5*np.max(img_lsst)
            vminroman=np.min(img_roman)
            vmaxroman= 1.5*np.max(img_roman)
        print(vmaxlsst)

        img_lsst, coords_lss, [ps_var_g, ps_var_r, ps_var_i] = simulate_rgb(lsst, size=size, kwargs_numerics=kwargs_numerics,time=time, source_x=sn_x,source_y=sn_y,returnsqrt=False)
        img_roman, coords_roman, [ps_var_g, ps_var_r, ps_var_i] = simulate_rgb(roman, size=size, kwargs_numerics=kwargs_numerics,time=time, source_x=sn_x,source_y=sn_y,returnsqrt=False)

        
        image = galsim.ImageF(img_lsst[0])
        affine_wcs = galsim.PixelScale(0.2).affine().withOrigin(image.center)
        ra = 0.0
        dec = 0.0
        wcs_LSST = galsim.TanWCS(affine_wcs,world_origin=galsim.CelestialCoord(ra*galsim.degrees,dec*galsim.degrees))
      
        ## psf = galsim.Image(psf_lsst[0,pad_lsst:-pad_lsst,pad_lsst:-pad_lsst])
        ## from scipy.signal import resample as interp
        ## psf_lsst_data = psf_lsst[0,pad_lsst:-pad_lsst,pad_lsst:-pad_lsst]
        ## psf_LS4_resample_shape = int(psf_lsst.shape[0]/5)
        ## psf_LS4_data = interp(psf_lsst,psf_LS4_resample_shape)[0]#,psf_LS4_resample_shape)) 
    
        ## # psf.wcs = wcs_LSST
        ## #psf.write(storedir+'/psf_LSST'+str(ind)+'_newSN.fits')

        image = galsim.ImageF(img_roman[0])
        affine_wcs2 = galsim.PixelScale(0.11).affine().withOrigin(image.center)
        ra = 0.0
        dec = 0.0
        wcs_Roman = galsim.TanWCS(affine_wcs2,world_origin=galsim.CelestialCoord(ra*galsim.degrees,dec*galsim.degrees))

        ## psf = galsim.Image(psf_roman[0,pad_roman:-pad_roman,pad_roman:-pad_roman])
        ## image_epsf = galsim.ImageF(psf_roman[0,pad_roman:-pad_roman,pad_roman:-pad_roman].shape[0],psf_roman[0,pad_roman:-pad_roman,pad_roman:-pad_roman].shape[1])
        ## psf.wcs = wcs_Roman
        ## #psf.write(storedir+'/'+str(ind)+'psf_Roman_'+str(ind)+'_newSN.fits')
        
        ## psf = galsim.Image(psf_LS4[0,pad_LS4:-pad_LS4,pad_LS4:-pad_LS4])
        ## image_epsf = galsim.ImageF(psf_LS4[0,pad_LS4:-pad_LS4,pad_LS4:-pad_LS4].shape[0],psf_LS4[0,pad_LS4:-pad_LS4,pad_LS4:-pad_LS4].shape[1])
        ## psf.wcs = wcs_LS4
        ## psf.write('/home/cw1074/ZTF/lensing/psf_LS4.fits')


z lens 0.05380463098441916 z source 0.7200284346083838
TIME  56100
pixel scale: 0.2
psf pixel_size: None
psf fwhm: 0.77
psf truncation: 5
DELAYS [0.         3.25901374] 3.2590137441131444


/home/vk9342/.conda/envs/usrp/lib/python3.11/site-packages/lenstronomy/LensModel/lens_model.py:96: UserWarning: Astropy Cosmology is provided. Make sure your cosmology model is consistent with the cosmology_model argument.
  warnings.warn(
/tmp/ipykernel_1450540/2042616440.py:20: RuntimeWarning: divide by zero encountered in log10
  magnitude = zero_points - 2.5 * np.log10(flux)
/tmp/ipykernel_1450540/2042616440.py:20: RuntimeWarning: divide by zero encountered in log10
  magnitude = zero_points - 2.5 * np.log10(flux)
/tmp/ipykernel_1450540/2042616440.py:20: RuntimeWarning: divide by zero encountered in log10
  magnitude = zero_points - 2.5 * np.log10(flux)
/tmp/ipykernel_1450540/2042616440.py:20: RuntimeWarning: divide by zero encountered in log10
  magnitude = zero_points - 2.5 * np.log10(flux)
/tmp/ipykernel_1450540/2042616440.py:20: RuntimeWarning: divide by zero encountered in log10
  magnitude = zero_points - 2.5 * np.log10(flux)
/tmp/ipykernel_1450540/2042616440.py:20: RuntimeWa

TypeError: unsupported operand type(s) for /: 'float' and 'NoneType'

In [ ]:
# %run simulating_different_telescopes-SNsims_multiepoch.py

In [89]:
## Get size of PSF
## code from: https://github.com/lenstronomy/lenstronomy/blob/main/lenstronomy/Data/psf.py
from lenstronomy.Util import util, kernel_util

def kernel_point_source(fwhm, pixel_size, truncation=5):
    sigma = util.fwhm2sigma(fwhm)
    # This num_pix definition is equivalent to that of the scipy ndimage.gaussian_filter
    # num_pix = 2r + 1 where r = round(truncation * sigma) is the radius of the gaussian kernel
    # kernel_num_pix is always an odd integer between 3 and 221
    kernel_radius = max(
        round(truncation * sigma / pixel_size), 1
    )
    kernel_num_pix = min(2 * kernel_radius + 1, 221)
    
    kernel_point_source = kernel_util.kernel_gaussian(
        kernel_num_pix, pixel_size, fwhm
    )
    return kernel_num_pix

# Rubin g-band
n_rubin_g = kernel_point_source(fwhm=0.77, pixel_size=0.2)
print(f"Rubin g-band n_psf: {n_rubin_g}")

# Roman g-band (F062)
n_roman_g = kernel_point_source(fwhm=0.058, pixel_size=0.11)
print(f"Roman g-band n_psf: {n_roman_g}")

Rubin g-band n_psf: 17
Roman g-band n_psf: 3
